In [ ]:
import sys
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from PyQt5.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QVBoxLayout, QHBoxLayout,
    QGridLayout, QTextEdit, QFileDialog, QComboBox, QSlider, QScrollArea,
    QFrame
)
from PyQt5.QtGui import QPixmap, QImage, QFont
from PyQt5.QtCore import Qt, QThread, pyqtSignal
from PIL import Image

# =========================
# Load SD model once globally
# =========================
print("Loading Stable Diffusion model...")
pipe = StableDiffusionPipeline.from_pretrained(
    "D:/models/sd_v1_5_local",
    torch_dtype=torch.float32,
    local_files_only=True
).to("cuda")
pipe.safety_checker = None

pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    "D:/models/sd_v1_5_local",
    torch_dtype=torch.float32,
    local_files_only=True
).to("cuda")
pipe_img2img.safety_checker = None
print("Model loaded!")

# =========================
# Worker Thread for generation
# =========================
class ImageGeneratorThread(QThread):
    finished = pyqtSignal(list)
    error = pyqtSignal(str)

    def __init__(self, prompt, steps, width, height, num_images, style_text, init_image=None):
        super().__init__()
        self.prompt = prompt
        self.steps = steps
        self.width = width
        self.height = height
        self.num_images = num_images
        self.style_text = style_text
        self.init_image = init_image

    def run(self):
        try:
            final_prompt = self.prompt
            if self.style_text:
                final_prompt += ", " + self.style_text
            if self.init_image:
                images = pipe_img2img(
                    prompt=final_prompt,
                    init_image=self.init_image,
                    strength=0.7,
                    num_inference_steps=self.steps,
                    num_images_per_prompt=self.num_images
                ).images
            else:
                images = pipe(
                    final_prompt,
                    num_inference_steps=self.steps,
                    height=self.height,
                    width=self.width,
                    num_images_per_prompt=self.num_images
                ).images
            self.finished.emit(images)
        except Exception as e:
            self.error.emit(str(e))

# =========================
# Main App
# =========================
class AIImageApp(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("AI Image Generator")
        self.setGeometry(150, 50, 1000, 900)
        self.setStyleSheet("""
            QWidget { background-color: #121212; color: #E0E0E0; font-family: Segoe UI; }
            QLabel { color: #E0E0E0; font-size: 13px; }
            QComboBox { background-color: #1e1e1e; color: white; border: 1px solid #555; border-radius: 8px; padding: 5px; }
            QComboBox QAbstractItemView { background-color: #1e1e1e; color: white; selection-background-color: #4CAF50; }
            QSlider::groove:horizontal { background: #444; height: 6px; border-radius: 3px; }
            QSlider::handle:horizontal { background: #4CAF50; width: 14px; margin: -4px 0; border-radius: 7px; }
            QScrollArea { border: none; }
        """)

        self.generated_images = []
        self.selected_image_index = 0
        self.init_image = None

        main_layout = QVBoxLayout()
        main_layout.setSpacing(15)
        main_layout.setContentsMargins(20, 20, 20, 20)

        # -------------------------
        # Prompt Box
        # -------------------------
        self.prompt_input = QTextEdit()
        self.prompt_input.setPlaceholderText("Enter your prompt here...")
        self.prompt_input.setFont(QFont("Segoe UI", 14))
        self.prompt_input.setMinimumHeight(130)
        self.prompt_input.setStyleSheet("""
            QTextEdit { padding: 15px; border-radius: 12px; border: 2px solid #555; background-color: #1e1e1e; color: white; }
        """)
        main_layout.addWidget(self.prompt_input)

        # -------------------------
        # Controls Row
        # -------------------------
        settings_frame = QFrame()
        settings_frame.setStyleSheet("QFrame { background-color: #1e1e1e; border-radius: 15px; padding: 20px; }")
        controls_layout = QGridLayout()
        controls_layout.setSpacing(20)
        controls_layout.setColumnStretch(1, 1)

        # ---------- Steps ----------
        steps_label = QLabel("Steps")
        steps_label.setStyleSheet("font-weight: 600; font-size: 23px; color: #FFFFFF;")
        self.steps_slider = QSlider(Qt.Horizontal)
        self.steps_slider.setMinimum(10)
        self.steps_slider.setMaximum(50)
        self.steps_slider.setValue(25)
        self.steps_slider.setTickInterval(5)
        self.steps_slider.setTickPosition(QSlider.NoTicks)
        self.steps_value = QLabel("25")
        self.steps_value.setStyleSheet("color: #4CAF50; font-weight: bold;")
        self.steps_slider.valueChanged.connect(lambda v: self.steps_value.setText(str(v)))
        steps_layout = QHBoxLayout()
        steps_layout.addWidget(self.steps_slider)
        steps_layout.addWidget(self.steps_value)
        controls_layout.addWidget(steps_label, 0, 0)
        controls_layout.addLayout(steps_layout, 0, 1)

        # ---------- Size ----------
        size_label = QLabel("Image Size")
        size_label.setStyleSheet("font-weight: 600; font-size: 23px; color: #FFFFFF;")
        self.size_dropdown = QComboBox()
        self.size_dropdown.addItems(["512x512", "512x768", "768x512", "768x768"])
        controls_layout.addWidget(size_label, 1, 0)
        controls_layout.addWidget(self.size_dropdown, 1, 1)

        # ---------- Style ----------
        style_label = QLabel("Style Preset")
        style_label.setStyleSheet("font-weight: 600; font-size: 23px; color: #FFFFFF;")
        self.style_dropdown = QComboBox()
        self.style_dropdown.addItems(["None", "Realistic", "Anime", "Cyberpunk", "Cinematic"])
        controls_layout.addWidget(style_label, 2, 0)
        controls_layout.addWidget(self.style_dropdown, 2, 1)

        # ---------- Images ----------
        images_label = QLabel("Images to Generate")
        images_label.setStyleSheet("font-weight: 600; font-size: 23px; color: #FFFFFF;")
        self.num_images_dropdown = QComboBox()
        self.num_images_dropdown.addItems(["1", "2", "3", "4"])
        controls_layout.addWidget(images_label, 3, 0)
        controls_layout.addWidget(self.num_images_dropdown, 3, 1)

        settings_frame.setLayout(controls_layout)
        main_layout.addWidget(settings_frame)

        # -------------------------
        # Image-to-Image Upload
        # -------------------------
        img2img_layout = QHBoxLayout()
        self.upload_button = QPushButton("Upload Image for Variants")
        self.upload_button.clicked.connect(self.upload_image)
        img2img_layout.addWidget(self.upload_button)
        main_layout.addLayout(img2img_layout)

        # -------------------------
        # Buttons
        # -------------------------
        btn_layout = QHBoxLayout()
        self.generate_button = QPushButton("Generate Image(s)")
        self.download_button = QPushButton("Download Image")
        self.download_button.setEnabled(False)
        for btn in [self.generate_button, self.download_button, self.upload_button]:
            btn.setMinimumHeight(50)
            btn.setFont(QFont("Segoe UI", 12, QFont.Bold))
            btn.setStyleSheet("""
                QPushButton { background-color: #4CAF50; color: white; border-radius: 12px; padding: 10px; }
                QPushButton:hover { background-color: #45a049; }
                QPushButton:pressed { background-color: #3e8e41; }
                QPushButton:disabled { background-color: gray; }
            """)
        self.generate_button.clicked.connect(self.start_generation)
        self.download_button.clicked.connect(self.download_image)
        btn_layout.addWidget(self.generate_button)
        btn_layout.addWidget(self.download_button)
        main_layout.addLayout(btn_layout)

        # -------------------------
        # Scrollable Gallery
        # -------------------------
        self.scroll_area = QScrollArea()
        self.scroll_area.setWidgetResizable(True)
        self.scroll_content = QWidget()
        self.scroll_layout = QGridLayout()
        self.scroll_layout.setSpacing(15)
        self.scroll_layout.setContentsMargins(10, 10, 10, 10)
        self.scroll_content.setLayout(self.scroll_layout)
        self.scroll_area.setWidget(self.scroll_content)
        main_layout.addWidget(self.scroll_area)

        self.setLayout(main_layout)

    # -------------------------
    # Upload image for img2img
    # -------------------------
    def upload_image(self):
        file_path, _ = QFileDialog.getOpenFileName(self, "Select Image", "", "Image Files (*.png *.jpg *.jpeg)")
        if file_path:
            self.init_image = Image.open(file_path).convert("RGB")
            self.image_label_temp = QLabel("Uploaded image ready for variants")
            self.image_label_temp.setStyleSheet("color:white;")
            self.scroll_layout.addWidget(self.image_label_temp)

    # -------------------------
    # Start Generation
    # -------------------------
    def start_generation(self):
        prompt = self.prompt_input.toPlainText().strip()
        if not prompt:
            return
        steps = self.steps_slider.value()
        size_text = self.size_dropdown.currentText()
        width, height = map(int, size_text.split("x"))
        num_images = int(self.num_images_dropdown.currentText())
        style = self.style_dropdown.currentText()
        style_texts = {
            "None": "",
            "Realistic": "photorealistic, high detail, 8k",
            "Anime": "anime style, vibrant colors, sharp outlines",
            "Cyberpunk": "cyberpunk, neon lights, futuristic city",
            "Cinematic": "cinematic lighting, film grain, epic composition"
        }
        style_text = style_texts.get(style, "")

        # Button states
        self.generate_button.setText("Generating...")
        self.generate_button.setEnabled(False)
        self.download_button.setEnabled(False)

        # Clear previous gallery
        for i in reversed(range(self.scroll_layout.count())):
            self.scroll_layout.itemAt(i).widget().setParent(None)

        self.thread = ImageGeneratorThread(prompt, steps, width, height, num_images, style_text, self.init_image)
        self.thread.finished.connect(self.display_images)
        self.thread.error.connect(self.show_error)
        self.thread.start()

    # -------------------------
    # Display images in scrollable gallery
    # -------------------------
    def display_images(self, images):
        self.generated_images = images
        self.selected_image_index = 0
        self.generate_button.setText("Generate Image(s)")
        self.generate_button.setEnabled(True)
        self.download_button.setEnabled(True)

        for idx, img in enumerate(images):
            pixmap = self.pil2pixmap(img)
            lbl = QLabel()
            lbl.setPixmap(pixmap.scaled(256, 256, Qt.KeepAspectRatio, Qt.SmoothTransformation))
            lbl.setFrameStyle(QFrame.Panel | QFrame.Raised)
            lbl.mousePressEvent = lambda e, i=idx: self.select_image(i)
            row = idx // 2
            col = idx % 2
            self.scroll_layout.addWidget(lbl, row, col)

    # -------------------------
    # Select image from gallery
    # -------------------------
    def select_image(self, index):
        self.selected_image_index = index

    # -------------------------
    # Convert PIL to QPixmap
    # -------------------------
    def pil2pixmap(self, image):
        data = image.tobytes("raw", "RGB")
        qimg = QImage(data, image.width, image.height, 3 * image.width, QImage.Format_RGB888)
        return QPixmap.fromImage(qimg)

    # -------------------------
    # Download selected image
    # -------------------------
    def download_image(self):
        if not self.generated_images:
            return
        if len(self.generated_images) == 1:
            file_path, _ = QFileDialog.getSaveFileName(self, "Save Image", "", "PNG Files (*.png)")
            if file_path:
                self.generated_images[0].save(file_path)
        else:
            folder_path = QFileDialog.getExistingDirectory(self, "Select Folder to Save Images")
            if folder_path:
                for i, img in enumerate(self.generated_images):
                    img.save(f"{folder_path}/generated_{i+1}.png")

    # -------------------------
    # Show errors
    # -------------------------
    def show_error(self, message):
        self.generate_button.setText("Generate Image(s)")
        self.generate_button.setEnabled(True)
        print("Error:", message)

# =========================
# Run App
# =========================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = AIImageApp()
    window.show()
    sys.exit(app.exec_())

C:\Users\USER\anaconda3\envs\sd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Stable Diffusion model...


Loading weights:   1%|▏               | 4/396 [00:00<00:14, 27.35it/s, Materializing param=special_care_embeds_weights]
Loading weights:   1%| | 5/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.class_
Loading weights:   1%| | 5/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.class_
Loading weights:   2%| | 6/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.patch_
Loading weights:   2%| | 6/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.patch_
Loading weights:   2%| | 7/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.positi
Loading weights:   2%| | 7/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.embeddings.positi
Loading weights:   2%| | 8/396 [00:00<00:14, 27.35it/s, Materializing param=vision_model.vision_model.encoder.layers.0.
Loading weights:   2%| | 8/396 [00:00<00

Model loaded!


 48%|███████████████████████████████████████▎                                          | 12/25 [02:41<02:42, 12.49s/it]